# Chapter 7: Scripting Fundamentals for Automation

## 1. Introduction

Welcome to the next step in building powerful automation scripts. So far, our scripts have been static, running the same way every time. In this section, you will learn to pass data directly into your scripts using command-line arguments. This fundamental technique transforms a script from a fixed procedure into a flexible tool.

Think of a script as a clinical protocol for a lab instrument. A static script can only run the protocol on a pre-defined sample. By using command-line arguments, you can tell the protocol which specific patient sample to analyze, what tests to run, and where to save the results, all without ever modifying the script's code. This makes your scripts reusable, scalable, and essential for automating real-world bioinformatics and health data tasks. By the end of this section, you will be able to build scripts that can adapt to different inputs on the fly.

---

## 2. Key Concepts and Definitions

Understanding these core terms is crucial for working with command-line arguments.

*   **Command-line Argument**: Data provided to a script when it is executed from the terminal. In a medical context, an argument could be a patient ID (`PT012345`), a file path to genomic data (`/data/patient_x.vcf`), or a setting for an analysis (`--quality-threshold 30`).
*   **Positional Parameter (`$1`, `$2`, ...)**: Special variables inside a script that hold the arguments in the order they are provided. `$1` always refers to the first argument, `$2` to the second, and so on. This is like processing a patient form where the first field is always the patient's ID and the second is their date of birth.
*   **`$0`**: A special variable containing the name of the script itself. This is useful for logging and error messages, allowing a script to report its own name, similar to how a medical report header identifies the procedure performed.
*   **`$#`**: A special variable that stores the total count of command-line arguments passed to the script. This is a critical tool for validation, like a lab technician verifying they have received the correct number of samples for a batch analysis before starting.
*   **`"$@"`**: A special variable that expands to all arguments as individual, quoted strings. This is the gold standard for iterating through a list of inputs, such as multiple patient files, because it correctly handles arguments that contain spaces (e.g., file names like `"Patient Report - John Doe.pdf"`).
*   **`shift`**: A built-in command that discards the first positional parameter (`$1`) and moves all subsequent parameters down by one (`$2` becomes `$1`, `$3` becomes `$2`, etc.). This is analogous to a clinical triage queue: after handling the first patient, you `shift` the queue to bring the next patient to the front.
*   **Parameter Expansion (`${1:-default}` or `${1:?error}`)**: A Bash feature to handle missing arguments gracefully. You can provide a default value if an argument is omitted (`${2:-"summary"}`) or exit the script with an error if a required argument is missing (`${1:?"Patient ID is required."}`).

---

## 3. Main Content

Here we explore the core techniques for using command-line arguments in your Bash scripts.

### 3.1 Accessing Arguments with Positional Parameters

You can access data passed to a script using positional parameters like `$1` for the first argument, `$2` for the second, and so on. The special variable `$0` holds the script's name, and `$#` holds the total number of arguments.

In [ ]:
%%bash
#!/bin/bash
# Displays patient info from command-line arguments.
echo "Script being run: $0"
echo "Fetching data for Patient ID: $1"
echo "Requested measurement: $2"
echo "Total arguments provided: $#"

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




*   **Execution**: Run the script with arguments: `./patient_info.sh PT012345 "blood pressure"`.
*   **Explanation**:
    *   `$0` becomes `./patient_info.sh`.
    *   `$1` becomes `PT012345`.
    *   `$2` becomes `blood pressure`. Note the quotes are necessary to treat it as a single argument.
    *   `$#` becomes `2`.
*   **Output**:
    ```
    Script being run: ./patient_info.sh
    Fetching data for Patient ID: PT012345
    Requested measurement: blood pressure
    Total arguments provided: 2
    ```

### 3.2 Handling Missing Arguments: Defaults and Errors

Parameter expansion allows you to build more robust scripts by setting default values for optional arguments or ensuring that required arguments are present.

In [ ]:
%%bash
#!/bin/bash
# Usage: ./generate_report.sh <patient_id> [report_type]
# Use ${1:?"Error"} to require an argument and exit if missing.
PATIENT_ID="${1:?"Error: Patient ID is required."}"
# Use ${2:="summary"} to set a fallback value if the second argument is omitted.
REPORT_TYPE="${2:="summary"}"
echo "Generating report for Patient ID: ${PATIENT_ID}"
echo "Report Type: ${REPORT_TYPE}"

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




> **Important:**
> Using `${1:?"Error"}` is a critical safety feature. It ensures your script doesn't proceed with missing or invalid data, which could otherwise lead to incorrect calculations, misidentified patient records, or failed processing downstream. Always enforce required inputs.

*   Running `./generate_report.sh PT007890` uses the default value for `REPORT_TYPE`, as the second argument is missing.
    ```
    Generating report for Patient ID: PT007890
    Report Type: summary
    ```

### 3.3 Processing Arguments Sequentially with `shift`

The `shift` command is perfect for processing a list of arguments one by one. It discards `$1` and re-indexes all remaining arguments.

> **In Practice:**
> The `shift` command mirrors the process of triaging a patient queue. You address the most urgent case (`$1`), and then `shift` your focus to the next person in line. This is useful for scripts that process a list of items sequentially, such as updating a batch of patient records or processing a series of lab results.

In [ ]:
%%bash
#!/bin/bash
# Usage: ./process_diagnoses.sh <primary_code> [secondary_codes...]
PRIMARY_DIAGNOSIS="$1"
echo "Primary Diagnosis: ${PRIMARY_DIAGNOSIS}"
shift # Discard the primary diagnosis; $@ now contains only secondary codes
# Loop through all remaining arguments (the secondary diagnoses)
i=1
while [ "$#" -gt 0 ]; do
echo "Processing Secondary Diagnosis #${i}: $1"
shift # Move to the next one
((i++))
done

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




*   Run: `./process_diagnoses.sh C50.9 Z85.3 I10`. The `shift` command allows the `while` loop to process each secondary diagnosis in turn.
    ```
    Primary Diagnosis: C50.9
    Processing Secondary Diagnosis #1: Z85.3
    Processing Secondary Diagnosis #2: I10
    ```

### 3.4 Iterating Through All Arguments: `"$@"` vs. `"$*"`

When you need to process every argument, especially when they might contain spaces, `"$@"` is the correct tool. It expands each argument into a separate, quoted string. In contrast, `"$*"` joins all arguments into a single string.

> **Debug Note:**
> A common bug is using `"$*"` when you mean `"$@"`. If your script is unexpectedly combining arguments or failing on file names with spaces, check that your `for` loop is using `for arg in "$@"`. This ensures each argument is treated as a separate and complete string.

In [ ]:
%%bash
#!/bin/bash
# Usage: ./process_meds.sh "lisinopril (Zestril) 10mg" "metformin (Glucophage) 500mg"
echo "--- Looping with \"\$@\" (preserves arguments):"
for arg in "$@"; do
echo "Argument: '$arg'"
done
echo ""
echo "--- Looping with \"\$*\" (joins arguments into one):"
for arg in "$*"; do
echo "Argument: '$arg'"
done

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




*   Running the script with quoted arguments demonstrates the critical difference:
    ```
    --- Looping with "$@" (preserves arguments):
    Argument: 'lisinopril (Zestril) 10mg'
    Argument: 'metformin (Glucophage) 500mg'

    --- Looping with "$*" (joins arguments into one):
    Argument: 'lisinopril (Zestril) 10mg metformin (Glucophage) 500mg'
    ```

---

## 4. Practice Exercises

Apply your knowledge with these hands-on exercises.

### Exercise 1: Simple Patient Greeting Basic

**Objective:** Practice accessing the first two positional parameters.
**Time:** 5 minutes
**Medical Context:** Creating a personalized script to greet a clinician and reference a patient file.

Write a script named `greeting.sh` that accepts a clinician's title and last name (e.g., "Dr." "Aoki") and a patient ID as arguments. It should print a personalized message.

*   Example Run: `./greeting.sh Dr. Aoki PT98765`
*   Expected Output: `Welcome, Dr. Aoki. You are now accessing the records for patient PT98765.`

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
#!/bin/bash
# Greets a clinician and states the patient ID being accessed.

# Check if enough arguments are provided
if [ "$#" -ne 3 ]; then
    echo "Usage: $0 <title> <last_name> <patient_id>" >&2
    exit 1
fi

TITLE="$1"
NAME="$2"
PATIENT_ID="$3"

echo "Welcome, ${TITLE} ${NAME}. You are now accessing the records for patient ${PATIENT_ID}."
```
**Explanation:** The script assigns `$1`, `$2`, and `$3` to descriptive variables. It then uses these variables in an `echo` statement to print the formatted message. An `if` statement is included to ensure the correct number of arguments are provided.
**Key Learning:** Accessing multiple positional parameters (`$1`, `$2`, `$3`) and using `$#` for basic argument validation.


</div>
</details>

### Exercise 2: Sample Processing Script Intermediate

**Objective:** Use parameter expansion to handle required and optional arguments.
**Time:** 10 minutes
**Medical Context:** A script to log the processing of a biological sample, where the sample ID is mandatory but the sequencing kit has a common default.

Write a script named `process_sample.sh` that accepts up to three arguments: a sample ID, a date (YYYY-MM-DD), and a sequencing kit name.
*   The script must exit with an error message `"Error: Sample ID is required."` if the first argument is not provided.
*   If the sequencing kit name (the third argument) is not provided, its value should default to `"Illumina MiSeq"`.
*   It must also print the total number of arguments it received.
*   Example Run: `./process_sample.sh SAM25-098 2025-11-11`
*   Expected Output:
    ```
    Processing sample SAM25-098 collected on 2025-11-11 with kit Illumina MiSeq.
    Total arguments received: 2
    ```
*   Running `./process_sample.sh` with no arguments should produce an error message like this (the script name and line number may vary):
    ```
    ./process_sample.sh: line 4: 1: Error: Sample ID is required.
    ```

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
#!/bin/bash
# Processes a sample with a required ID and optional kit name.

SAMPLE_ID="${1:?"Error: Sample ID is required."}"
COLLECTION_DATE="$2"
KIT_NAME="${3:-"Illumina MiSeq"}"

echo "Processing sample ${SAMPLE_ID} collected on ${COLLECTION_DATE} with kit ${KIT_NAME}."
echo "Total arguments received: $#"
```
**Explanation:** `${1:?"..."}` ensures the script fails if the sample ID is missing. `${3:-"..."}` provides a default value for the kit name if the third argument is not supplied. `$#` correctly reports the number of arguments that were actually passed on the command line.
**Key Learning:** Applying parameter expansion for both required (`:?`) and default (`:-`) argument handling.


</div>
</details>

### Exercise 3: Batch File Validator Advanced

**Objective:** Iterate through an unknown number of arguments using `"$@"` and perform a check on each.
**Time:** 15 minutes
**Medical Context:** Before running a large bioinformatics pipeline, it's crucial to verify that all input data files exist. This script automates that check.

Write a script named `validate_files.sh` that accepts one or more file paths as command-line arguments. The script should loop through each argument and check if it exists as a file.
*   For each argument, it should print `SUCCESS: <file_path> exists.` or `ERROR: <file_path> not found.`.
*   To test, first create some dummy files: `touch patient_A.csv patient_C.csv`.
*   Example Run: `./validate_files.sh patient_A.csv patient_B.csv patient_C.csv`
*   Expected Output:
    ```
    SUCCESS: patient_A.csv exists.
    ERROR: patient_B.csv not found.
    SUCCESS: patient_C.csv exists.
    ```

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
#!/bin/bash
# Validates the existence of multiple files passed as arguments.

# Check if at least one argument is given
if [ "$#" -eq 0 ]; then
    echo "Usage: $0 <file1> [file2] ..." >&2
    exit 1
fi

echo "Starting file validation..."
for file in "$@"; do
    if [ -f "$file" ]; then
        echo "SUCCESS: $file exists."
    else
        echo "ERROR: $file not found."
    fi
done
```
**Explanation:** The script first checks if any arguments were provided. Then, the `for file in "$@"` loop iterates through each argument passed to the script. The `[ -f "$file" ]` test condition checks if a file with that name exists and is a regular file.
**Key Learning:** Using the `for` loop with `"$@"` to robustly handle an arbitrary number of arguments, a common pattern for batch processing scripts.


</div>
</details>

> **Reflection Moment:**
> In the sample processing lab scenario, what other arguments might be useful? Could you add an optional argument for the researcher's name with a default value of "unassigned"? How would you modify the script to handle that?

---

## 5. Practical Applications

Command-line arguments are the bridge between your scripts and real-world, dynamic data. Here are some practical applications in precision health.

*   **Batch Processing of Genomic Data**: A bioinformatician can write a script to annotate genetic variants. Instead of running it manually for hundreds of patient VCF (Variant Call Format) files, they can execute `./annotate_variants.sh /data/vcf/*.vcf`. The script uses a `for file in "$@"` loop to apply the same annotation logic to every file, saving hours of manual work and ensuring consistency.
*   **Generating Custom Patient Reports**: A clinician needs a summary of a patient's lab results from the last month. A script can be run as `./generate_report.sh PT12345 2025-10-01 2025-11-01`. The script takes the patient ID (`$1`), start date (`$2`), and end date (`$3`) to query a database, format the results, and generate a PDF, providing on-demand, customized clinical summaries.
*   **Automating Quality Control on Sequencing Runs**: After a DNA sequencer completes a run, a script can be automatically triggered: `./qc_pipeline.sh --run-id RUN-55AB --kit-type "TruSeq" --output-dir /results/RUN-55AB`. This script uses arguments to know which run data to analyze, what quality metrics to apply based on the kit, and where to store the output reports, enabling a fully automated "sequencer-to-report" pipeline.
*   **Dynamic Environment Configuration for Clinical Software**: A script that launches a clinical analysis application can accept arguments to configure its environment. For instance, `./launch_app.sh --mode "clinical" --user "Dr.Smith"` could set the application to run with FDA-compliant features enabled (`clinical` mode) and log all actions under Dr. Smith's user profile, ensuring auditability and compliance.

---

## 6. Summary and Key Takeaways

In this section, we've explored how to make Bash scripts interactive and reusable by using command-line arguments. You learned to capture, validate, and process external data, transforming your scripts from simple command sequences into powerful, flexible tools.

*   You can access arguments passed to a script using positional parameters: `$1`, `$2`, and so on.
*   Special variables provide critical metadata: `$#` gives the count of arguments, `$0` provides the script's name, and `"$@"` represents all arguments as a list of separate items.
*   Robust scripts should handle missing arguments, either by providing default values with `${VAR:-default}` or by exiting with an error for required ones with `${VAR:?message}`.
*   For processing a list of items, such as batch-processing files, the `for arg in "$@"` loop is the standard and safest method.
*   The `shift` command is a useful technique for processing arguments sequentially, especially when handling a primary argument differently from subsequent ones.

With this foundation, you are now equipped to build scripts that adapt to different inputs. In the next section, we will delve into **conditional logic**, allowing your scripts to make decisions and change their behavior based on the arguments they receive.

---


---

## 📝 Interactive Practice

Practice the concepts with these interactive exercises:

### script using positional parameters like `$1` for the first argument, `$2` for the second, and so on. The special variable `$0` holds the script's name, and `$#` holds the total number of arguments.

In [ ]:
%%bash
#!/bin/bash
# Displays patient info from command-line arguments.
echo "Script being run: $0"
echo "Fetching data for Patient ID: $1"
echo "Requested measurement: $2"
echo "Total arguments provided: $#"

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### Parameter expansion allows you to build more robust scripts by setting default values for optional arguments or ensuring that required arguments are present.

In [ ]:
%%bash
#!/bin/bash
# Usage: ./generate_report.sh <patient_id> [report_type]
# Use ${1:?"Error"} to require an argument and exit if missing.
PATIENT_ID="${1:?"Error: Patient ID is required."}"
# Use ${2:="summary"} to set a fallback value if the second argument is omitted.
REPORT_TYPE="${2:="summary"}"
echo "Generating report for Patient ID: ${PATIENT_ID}"
echo "Report Type: ${REPORT_TYPE}"

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### shift` your focus to the next person in line. This is useful for scripts that process a list of items sequentially, such as updating a batch of patient records or processing a series of lab results.

In [ ]:
%%bash
#!/bin/bash
# Usage: ./process_diagnoses.sh <primary_code> [secondary_codes...]
PRIMARY_DIAGNOSIS="$1"
echo "Primary Diagnosis: ${PRIMARY_DIAGNOSIS}"
shift # Discard the primary diagnosis; $@ now contains only secondary codes
# Loop through all remaining arguments (the secondary diagnoses)
i=1
while [ "$#" -gt 0 ]; do
echo "Processing Secondary Diagnosis #${i}: $1"
shift # Move to the next one
((i++))
done

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### unexpectedly combining arguments or failing on file names with spaces, check that your `for` loop is using `for arg in "$@"`. This ensures each argument is treated as a separate and complete string.

In [ ]:
%%bash
#!/bin/bash
# Usage: ./process_meds.sh "lisinopril (Zestril) 10mg" "metformin (Glucophage) 500mg"
echo "--- Looping with \"\$@\" (preserves arguments):"
for arg in "$@"; do
echo "Argument: '$arg'"
done
echo ""
echo "--- Looping with \"\$*\" (joins arguments into one):"
for arg in "$*"; do
echo "Argument: '$arg'"
done

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
#!/bin/bash
# Greets a clinician and states the patient ID being accessed.
# Check if enough arguments are provided
if [ "$#" -ne 3 ]; then
echo "Usage: $0 <title> <last_name> <patient_id>" >&2
exit 1
fi
TITLE="$1"
NAME="$2"
PATIENT_ID="$3"
echo "Welcome, ${TITLE} ${NAME}. You are now accessing the records for patient ${PATIENT_ID}."

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
#!/bin/bash
# Processes a sample with a required ID and optional kit name.
SAMPLE_ID="${1:?"Error: Sample ID is required."}"
COLLECTION_DATE="$2"
KIT_NAME="${3:-"Illumina MiSeq"}"
echo "Processing sample ${SAMPLE_ID} collected on ${COLLECTION_DATE} with kit ${KIT_NAME}."
echo "Total arguments received: $#"

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
#!/bin/bash
# Validates the existence of multiple files passed as arguments.
# Check if at least one argument is given
if [ "$#" -eq 0 ]; then
echo "Usage: $0 <file1> [file2] ..." >&2
exit 1
fi
echo "Starting file validation..."
for file in "$@"; do
if [ -f "$file" ]; then
echo "SUCCESS: $file exists."
else
echo "ERROR: $file not found."
fi
done

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above


